# 03 — Cardiovascular Risk Modelling

**P.U.L.S.E.**

XGBoost classifier on the UCI Statlog Heart dataset (270 patients, 13 features).

Covers:
- Feature importance & SHAP beeswarm
- Bootstrap confidence interval on AUC (1 000 resamples)
- Confusion matrix at optimal F1 threshold
- MLflow run inspection

In [ ]:
import os, sys
ROOT = os.path.dirname(os.path.abspath('.'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, f1_score
)
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid', palette='muted')
MODEL_DIR = os.path.join(ROOT, 'models')
print('Ready')

## 1  Load model & data

In [ ]:
with open(os.path.join(MODEL_DIR, 'cardio_xgb.pkl'), 'rb') as f:
    model = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'cardio_features.pkl'), 'rb') as f:
    features = pickle.load(f)

from src.data.loader import load_statlog_heart
df = load_statlog_heart()
# target: 1=absent, 2=present  →  0/1
y  = (df['target'] == 2).astype(int)
X  = df[features]

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
proba = model.predict_proba(X_test)[:, 1]
print(f'Test AUC: {roc_auc_score(y_test, proba):.4f}  |  n_test={len(X_test)}')

## 2  ROC + bootstrap CI

In [ ]:
# Bootstrap AUC CI
rng  = np.random.default_rng(0)
boot = [
    roc_auc_score(
        y_test.values[idx := rng.integers(0, len(y_test), len(y_test))],
        proba[idx]
    )
    for _ in range(1_000)
]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f'Bootstrap 95% CI: [{lo:.4f}, {hi:.4f}]')

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, proba, ax=ax, name='Cardio XGBoost')
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_title(f'Cardio ROC — AUC={roc_auc_score(y_test, proba):.3f}  CI[{lo:.3f},{hi:.3f}]')
plt.tight_layout()
plt.show()

## 3  Confusion matrix at optimal threshold

In [ ]:
thresholds = np.linspace(0.1, 0.9, 80)
f1s = [f1_score(y_test, (proba >= t).astype(int), zero_division=0) for t in thresholds]
opt_thr = thresholds[np.argmax(f1s)]
y_pred  = (proba >= opt_thr).astype(int)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease']).plot(
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title(f'Confusion Matrix (threshold={opt_thr:.2f}, F1={max(f1s):.3f})')
plt.tight_layout()
plt.show()

## 4  SHAP beeswarm

In [ ]:
explainer  = shap.TreeExplainer(model.get_booster())
shap_vals  = explainer.shap_values(X_test)

shap.summary_plot(shap_vals, X_test, show=False)
plt.title('SHAP Beeswarm — Cardio XGBoost')
plt.tight_layout()
plt.show()

## 5  Feature importance bar chart

In [ ]:
imp = pd.Series(
    np.abs(shap_vals).mean(axis=0), index=features
).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
imp.plot.barh(ax=ax, color=sns.color_palette('muted')[0])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Feature Importance — Cardio XGBoost')
plt.tight_layout()
plt.show()